# MailSense — dataset exploration

Inspection of the Enron-based actionable e-mail dataset before any modelling:
structure, label distribution, data-quality problems, text length, and the
vocabulary that separates the two classes.

Research question (PROJECT_SPEC.md §2): *does this e-mail content require the
recipient's action or attention?* Labels: `Yes` = Actionable, `No` = Non-Actionable.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.preprocessing.prepare_data import read_raw
from src.preprocessing.clean import clean_text

RAW = ROOT / 'data' / 'Ask0729-fixed.txt'
raw, malformed = read_raw(RAW)
print(f'parsed rows : {len(raw)}')
print(f'malformed   : {len(malformed)}')
raw.head()

## 1. Class distribution

Mild imbalance only, so accuracy is not meaningless — but precision, recall and
especially the false-negative count still carry the important information.

In [ ]:
counts = raw['label'].value_counts()
print(counts)
print((counts / counts.sum()).round(4))
counts.plot.bar(title='Label distribution', rot=0, color=['#4c72b0', '#dd8452']);

## 2. Data quality

Missing values, duplicates, conflicting duplicates and encoding damage.

In [ ]:
raw['text'] = raw['raw_text'].map(clean_text)

print('missing raw text      :', raw['raw_text'].isna().sum())
print('empty after cleaning  :', (raw['text'].str.len() == 0).sum())
print('exact duplicate texts :', raw.duplicated(subset=["text"]).sum())

conflict = raw.groupby('text')['label'].nunique()
conflict = conflict[conflict > 1]
print('texts with both labels:', len(conflict))

mojibake = raw[raw['raw_text'].str.contains('\ufffd', regex=False)]
print('rows with lost bytes  :', len(mojibake))
mojibake[['label', 'raw_text', 'text']].head(3)

## 3. Text length

These are sentences/snippets, not full e-mails. The 99th percentile decides the
`max_len` used by the LSTM and BERT.

In [ ]:
raw['n_words'] = raw['text'].str.split().map(len)
print(raw['n_words'].describe().round(2))
print(raw['n_words'].quantile([.5, .75, .9, .95, .99]))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
raw['n_words'].clip(upper=80).plot.hist(bins=40, ax=ax[0], title='Words per example')
raw.boxplot(column='n_words', by='label', ax=ax[1])
ax[1].set_ylim(0, 80); ax[1].set_title('Length by class'); plt.suptitle('');

## 4. What words signal actionability?

A quick log-odds view of the vocabulary. This is exploration only — nothing here
feeds back into the models, and the feature weights used for modelling are fitted
on the training split alone.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(min_df=5)
X = vec.fit_transform(raw['text'])
y = (raw['label'] == 'Yes').to_numpy()

pos = np.asarray(X[y].sum(axis=0)).ravel() + 1
neg = np.asarray(X[~y].sum(axis=0)).ravel() + 1
log_odds = np.log((pos / pos.sum()) / (neg / neg.sum()))
terms = np.array(vec.get_feature_names_out())

order = np.argsort(log_odds)
print('Most ACTIONABLE terms   :', ', '.join(terms[order[-25:]][::-1]))
print()
print('Most NON-ACTIONABLE terms:', ', '.join(terms[order[:25]]))

The actionable side is dominated by requests, second-person address and temporal
deadlines; the non-actionable side by promotional and informational language.
This is exactly why the preprocessing policy keeps stopwords, modals and
temporal expressions instead of stripping them.

## 5. Example snippets from each class

In [ ]:
for label in ['Yes', 'No']:
    print(f'--- {label} ---')
    for t in raw[raw['label'] == label]['text'].sample(6, random_state=42):
        print(' ', t[:140])
    print()

## 6. The saved splits

Produced once by `python -m src.preprocessing.prepare_data` and reused by all
three models, so every model sees the same training data and the same test set.

In [ ]:
report_path = ROOT / 'results' / 'data_quality_report.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    for k in ['final_rows', 'removed_total', 'train_size', 'val_size', 'test_size',
              'train_class_distribution', 'val_class_distribution',
              'test_class_distribution']:
        print(f'{k:28} {report[k]}')
else:
    print('Run:  python -m src.preprocessing.prepare_data')